

July 25th 2025

Want to explore whether recurrent inhibitory connections in a ring architecture can compensate for changes in input, leading to a more stereotyped output. We'll look at two cases:

(a) input bump amplitude changes

(b) multiple input bumps, possibly representing an ambiguous or rapidly changing head direction estimate in the EPGs.


These simulations use a simple abstract architecture where feedforward input is bump-shaped, recurrent inhibitory connectivity is from the ring network of Burak & Fiete, and f-I curve is threshold linear. Later make this architecturally closer to Delta7s and explore nonlinear f-I curves.


We'll be using divisive inhibition (i.e., divisive normalization). Thus, ignoring dynamics the rates in the network are given by
\begin{equation}
r_i = \frac{f(I)}{\alpha + W\vec{r}} = \frac{f(I)}{\alpha + \sum_j W_{ij}r_j},
\end{equation}
where $r_i$ is the rate of the $i$th neuron, $f$ is the f-I curve, $\alpha$ is a small constant and $W$ is the recurrent inhibitory connectivity matrix.



In [ ]:
import sys, os, glob
import datetime, time

import numpy as np
from numpy import linalg as nla
#import scipy.linalg as sla
import numpy.random as nrd

# import scipy.stats as sst

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

import pandas as pd

gen_fns_dir = os.path.abspath('../shared')
sys.path.append(gen_fns_dir)

import ring_net_fns as rnf

curr_date=datetime.datetime.now().strftime('%Y_%m_%d')+'_'
sd=int((time.time()%1)*(2**31))
rng = nrd.default_rng(sd)
print('Seed= ',sd)

# Define a custom colormap from dark red to light red
cm_red = LinearSegmentedColormap.from_list("light_to_dark_red", ["lightcoral", "red", "darkred"])

cm_blue = LinearSegmentedColormap.from_list("light_to_dark_blue", ["lightsteelblue", "blue", "darkblue"])




Set up simple models for the feedforward input and the recurrent inhibitory weights. Next round will use more realistic connections (i.e., EPG - Delta7 feedforward weights and Delta7-Delta7 recurrent weights)



In [ ]:
# External inputs are bumps of varying magnitude

def gauss_bump(theta, center=np.pi/2, amp=1.0, width=0.5):
    d = np.abs(theta - center)
    d = np.minimum(d, 2*np.pi - d)
    return amp * np.exp(-d**2 / (2*width**2))

def cosine_bump(theta, center=np.pi/2., amplitude=1.0, width=np.pi/4):
    """Cosine bump truncated below 0.

    width: half-period controlling support; we map angle diff to cosine over that width.
    """
    
    d = (theta - center + np.pi) % (2*np.pi) - np.pi
    x = np.clip(1 - (d/width)**2, 0, None)  # parabolic window -> smooth bump
    return amplitude * x


In [ ]:
theta=np.linspace(0, 2*np.pi, 100)
fig, ax = plt.subplots(1,1,figsize=(4,4))
ax.plot(theta, cosine_bump(theta), color='k')
ax.plot(theta, cosine_bump(theta, center=np.pi, amplitude=2.0, width=np.pi/2.), color='r')


In [ ]:
# For recurrent inhibition connectivity profile, just use the one from Burak and Fiete for now.
# Note that we plot the connectivity profile over two cycles so that we can see periodicity
d_theta = np.arange(-2*np.pi, 2*np.pi, 0.1)

fig, ax = plt.subplots(1,1,figsize=(6,4))
ax.plot(d_theta, rnf.bf_conn_profile(d_theta), lw=3)
ax.set(xlabel='Angle difference', ylabel='Connection strength')
# ax.set_xlim([-np.pi, np.pi])
plt.show()


In [ ]:
# Set up recurrent connections and some input
n_neurons = 300 
# Connectivity matrix, consisting of shifted copies of connectivity profile
theta, signed_W = rnf.get_W_and_pref_angles(n_neurons, rnf.bf_conn_profile)
# Note that signed_W has negative signs, that we don't need for divisive setting,
# so remove them.
# Also setting a scale factor of 0.5 because that seems to bring things close to 1,
# for convenience, but can change later.
W = 0.5*np.abs(signed_W)

# Template feedforward input for convenience
epg_input_template = cosine_bump(theta)

fig, ax = plt.subplots(1,2,figsize=(8,4))
ax[0].plot(theta, epg_input_template)
ax[0].set(xlabel='Neuron angle', ylabel='FF input')
ax[1].plot(theta, W[0])
ax[1].set(xlabel='Angle difference', ylabel='Recurrent weight')
plt.tight_layout()




Set up network with some simple dynamics. We'll use
\begin{equation}
\tau\frac{dr_i}{dt} = -r_i + \frac{f(I)}{\alpha + \sum_j W_{ij}r_j},
\end{equation}
where $r_i$ is the rate of the $i$th neuron, $\tau$ is a time-constant, $f$ is the f-I curve, $\alpha$ is a small constant and $W$ is the recurrent inhibitory connectivity matrix.



In [ ]:
# Activation functions
# Primarily using rectified linear
def relu(x):
    return np.maximum(x, 0)  # ReLU

# But keep the sigmoid around in case want to use it later in this notebook
def sigmoid(x, amplitude=1., slope=1., midpoint=0.):
    return amplitude / (1 + np.exp(-slope*(x-midpoint)))

# Simulation function
def simulate_ring(I, W, f, steps, dt, tau, alpha, ics):
    r = np.zeros((steps, W.shape[0]))
    r[0] = ics
    for i in range(steps-1):
        ff_input = f(I)
        rec_inh = W @ r[i]
        r_ss = ff_input / (alpha + rec_inh)  # divisive normalization
        r[i+1] = r[i] + dt/tau * (-r[i] + r_ss)
    return r




So right now there are effectively three explicit parameters: $\tau$, $\alpha$ and the amplitude of the feedforward input $I$. Also an implicit parameter in the scale of the recurrent inhibitory connections (and then the widths/shapes of the input and recurrent connections).





Before running dynamics, let's see what we'd get with feedforward divisive inhibition.
This should give us near-perfect cancellation.


It's also perhaps a good proxy for what the network initially sees: if the feedforward input quickly drives the Delta7s up to $f(I)$, then the recurrent inhibition initially looks like feedforward divisive inhibition and so tries to drive the network to the steady state given by feedforward divisive inhibition. As the firing rates change, though, the inhibition will change proportionally and so the system won't move to the same steady-state.



In [ ]:
alpha = 0.1
# Amplitude of EPG input
input_amplitudes = [1,5, 10, 20, 40]

results = []
max_vals = []

for amp in input_amplitudes:
    epg_input = amp * epg_input_template
    ff_inh = W @ epg_input
    r_ss = epg_input / (alpha + ff_inh)
    results.append(r_ss)
    max_vals.append(np.max(r_ss))

fig, ax = plt.subplots(1,3,figsize=(12,4))
for i, r in enumerate(results):
    ax[0].plot(theta, r, label=f"Amp = {input_amplitudes[i]}")
ax[0].set(xlabel='Neuron angle (rad)', ylabel='rate', title='Rates for diff. input amplitudes')    
ax[0].legend()
ax[1].plot(input_amplitudes, max_vals)
ax[1].set(xlabel='input amplitude', ylabel='max firing rate', title='Bump height for diff input amplitudes')
# Also get a sense of the profile of inhibition. Note that alpha gets added to this in the equation
ax[2].plot(theta, W @ epg_input_template)
ax[2].set(xlabel='Neuron angle (rad)', ylabel='inhibition', title='FF inhibition for bump with amplitude 1')
plt.tight_layout()
plt.show()




Now look at dynamics and see how things vary with input amplitude



In [ ]:
# First simulate for one amplitude to see if we get a bump.
# Note that bump shape depends on the f-I curve and can be changed. Even in the ReLU case, adding
# a constant offset will broaden bump. Currently bump is quite narrow.
# Regenerating input
epg_input = cosine_bump(theta)
ics = epg_input # Start network where FF input wants it to be (effectively assuming FF input is fast)
alpha = 0.1
n_steps = int(1e3)
rates = simulate_ring(I=epg_input, W=W, f=relu, steps=n_steps, dt=0.05, tau=1, alpha=alpha, ics=ics)

fig, ax = plt.subplots(1,1,figsize=(6,4))
ax.plot(theta, rates[0], color=cm_red(0), label='initial state')
ax.plot(theta, rates[40], color=cm_red(0.5), label='intermediate')
ax.plot(theta, rates[-1], color=cm_red(1.), label='end')
ax.legend()


In [ ]:
# Now look at a range of input amplitudes
epg_input_template = cosine_bump(theta) # Amplitude 1
alpha = 0.1
# Amplitude of EPG input
input_amplitudes = [1,5, 10, 20, 40]
results = []
max_vals = []

for amp in input_amplitudes:
    epg_input = amp * epg_input_template
    ics = epg_input
#     ics = epg_input_template
    rates = simulate_ring(I=epg_input, W=W, f=relu, steps=int(1e3), dt=0.05, tau=1, alpha=alpha, ics=ics)
    results.append(rates)
    max_vals.append(np.max(rates[-1]))
    
    
fig, ax = plt.subplots(2,2,figsize=(10,8))
for i, r in enumerate(results):
    ax[0,0].plot(theta, relu(input_amplitudes[i]*epg_input_template), color=cm_blue(i/len(results)))
    ax[0,1].plot(theta, r[-1], color=cm_blue(i/len(results)), label=f"Input amp = {input_amplitudes[i]}")
ax[0,0].set(xlabel='Neuron angle (rad)', ylabel='input', title='Steady-state rates without inhibition')    
ax[0,1].set(xlabel='Neuron angle (rad)', ylabel='rate', title='Steady-state rates with inhibition')    
ax[0,1].legend()
ax[1,0].plot(input_amplitudes, max_vals, label='inhibition')
ax[1,0].plot(input_amplitudes, input_amplitudes, ls='--', label='no inhibition')
ax[1,0].legend()
ax[1,0].set(xlabel='input amplitude', ylabel='max firing rate', title='Steady-state bump height for diff input amplitudes')

# Also get a sense of the profile of inhibition for some input. Note that alpha gets added to this in the equation
ax[1,1].plot(theta, W @ results[2][-1])
ax[1,1].set(xlabel='Neuron angle (rad)', ylabel='inhibition', title='Steady-state recurrent inhibition for bump with amplitude 10')
plt.tight_layout()
plt.savefig('linear_ring_with_div_norm_amplitudes.png')
plt.show()
    




Bump amplitude changes significantly, but in a much narrower range than without inhibition (note scale on plots).





We also want to look at situations where the input no longer provides a unique bump, perhaps because the bump is in the process of switching locations. In this case is the output still localized to a bump?



In [ ]:
epg_input_bimodal = cosine_bump(theta, amplitude=1.) + cosine_bump(theta, amplitude=0.4, center=3*np.pi/2.)
fig, ax = plt.subplots(1, 1, figsize=(6,4))
ax.plot(theta, epg_input_bimodal)
ax.set(xlabel='Neuron angle (rad)', ylabel='input', title='Multiple bump input') 


In [ ]:
alpha = 0.1
ics = epg_input_bimodal

rates = simulate_ring(I=epg_input_bimodal, W=W, f=relu, steps=int(2e3), dt=0.05, tau=1, alpha=alpha, ics=ics)

fig, ax = plt.subplots(1,1,figsize=(6,4))
ax.plot(theta, rates[0], color=cm_red(0), label='initial state')
ax.plot(theta, rates[10], color=cm_red(0.5), label='intermediate')
ax.plot(theta, rates[-1], color=cm_red(1.), label='end')
ax.legend()
plt.savefig('linear_ring_with_div_norm_multiple_bumps.png')
